# Phase 7 structured telemetry walkthrough

Use this notebook to inspect the Phase 7 run-event telemetry changes by hand. It exercises the real API routes, LangGraph orchestration, `RunEvent` schema, emitter wrapper, and JSONL sink, while using a scripted LLM and fake tool bindings so the examples are deterministic and do not require Postgres or RAG setup.

The examples focus on the debugging surfaces added in Phase 7:

- the stable `RunEvent` and `ToolCallSummary` schema
- one JSONL event for a normal completed `/agent/query` run
- one JSONL event for an unknown-asset completion
- two correlated JSONL events for a HITL pause/resume workflow
- one JSONL event for an internal error response

Start Jupyter from the project environment, then run the cells in order.

## 1. Configure imports and notebook helpers

The notebook writes local JSONL output under `notebooks/.phase7-run-events/` by default. Each scenario uses a separate file so you can inspect the raw lines between cells.

In [1]:
import json
import sys
from collections.abc import AsyncGenerator
from contextlib import asynccontextmanager
from datetime import UTC, date, datetime
from decimal import Decimal
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Literal, cast

from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient
from sqlalchemy.ext.asyncio import AsyncSession

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from maintenance_agent.api import agent as agent_api  # noqa: E402
from maintenance_agent.api.agent import router as agent_router  # noqa: E402
from maintenance_agent.db.repositories.records import AssetRecord, WorkOrderRecord  # noqa: E402
from maintenance_agent.llm.client import (  # noqa: E402
    LLMMessage,
    LLMResponse,
    LLMTool,
    LLMToolChoice,
    ToolCallRequest,
)
from maintenance_agent.orchestration import graph as graph_module  # noqa: E402
from maintenance_agent.orchestration.graph import AgentGraphDependencies, build_agent_graph  # noqa: E402
from maintenance_agent.orchestration.state import GraphState, WorkOrderDraft  # noqa: E402
from maintenance_agent.schemas.agent import AgentQueryResponse  # noqa: E402
from maintenance_agent.schemas.run_event import RunEvent, ToolCallSummary  # noqa: E402
from maintenance_agent.telemetry.run_events import make_jsonl_emitter, read_run_events  # noqa: E402
from maintenance_agent.tools.get_asset_status import ClassifiedReading, GetAssetStatusResult  # noqa: E402
from maintenance_agent.tools.get_maintenance_history import GetMaintenanceHistoryResult  # noqa: E402
from maintenance_agent.tools.get_plant_policy import GetPlantPolicyResult  # noqa: E402
from maintenance_agent.tools.resolve_asset import ResolveAssetResult  # noqa: E402
from maintenance_agent.tools.search_maintenance_docs import (  # noqa: E402
    DocSearchHit,
    SearchMaintenanceDocsResult,
)

In [2]:
RUN_EVENTS_DIR = PROJECT_ROOT / "notebooks" / ".phase7-run-events"
RUN_EVENTS_DIR.mkdir(parents=True, exist_ok=True)


def print_json(value):
    if hasattr(value, "model_dump"):
        value = value.model_dump(mode="json")
    print(json.dumps(value, indent=2, default=str))


def reset_event_file(name):
    path = RUN_EVENTS_DIR / f"{name}.jsonl"
    path.unlink(missing_ok=True)
    return path


def summarize_event(event):
    return {
        "event_id": str(event.event_id),
        "run_id": event.run_id,
        "emitted_at": event.emitted_at.isoformat(),
        "latency_ms": event.latency_ms,
        "status": event.status,
        "request": event.request,
        "tool_calls": [call.model_dump() for call in event.tool_calls],
        "final_output_status": event.final_output.status,
        "final_output_request_id": event.final_output.request_id,
        "pending_action": event.final_output.pending_action.model_dump(mode="json")
        if event.final_output.pending_action
        else None,
        "error": event.error.model_dump(mode="json") if event.error else None,
    }


def print_event_table(path):
    events = read_run_events(path)
    print_json([summarize_event(event) for event in events])
    return events

## 2. Inspect the Phase 7 schema

`RunEvent` is the stable ingestion contract. `ToolCallSummary` is deliberately thin: only the ordered tool name and sequence are stored outside the full `final_output` payload.

In [3]:
print_json(
    {
        "RunEvent": RunEvent.model_json_schema(),
        "ToolCallSummary": ToolCallSummary.model_json_schema(),
    }
)

{
  "RunEvent": {
    "$defs": {
      "AgentError": {
        "properties": {
          "code": {
            "title": "Code",
            "type": "string"
          },
          "message": {
            "title": "Message",
            "type": "string"
          }
        },
        "required": [
          "code",
          "message"
        ],
        "title": "AgentError",
        "type": "object"
      },
      "AgentQueryResponse": {
        "properties": {
          "request_id": {
            "title": "Request Id",
            "type": "string"
          },
          "status": {
            "enum": [
              "ok",
              "needs_approval",
              "insufficient_evidence",
              "unknown_asset",
              "error"
            ],
            "title": "Status",
            "type": "string"
          },
          "asset_id": {
            "anyOf": [
              {
                "type": "string"
              },
              {
                "type": "

## 3. Deterministic API harness

These helpers keep the notebook deterministic. The graph and API route layer remain real; only LLM outputs, tool bindings, and session acquisition are swapped so the notebook can run without a database.

In [4]:
class ScriptedLLMClient:
    def __init__(self, responses):
        self._responses = list(responses)
        self.calls = []

    async def generate(
        self,
        messages: list[LLMMessage],
        *,
        tools: list[LLMTool] | None = None,
        tool_choice: LLMToolChoice | None = None,
    ) -> LLMResponse:
        self.calls.append(
            {
                "messages": [message.model_dump(mode="json") for message in messages],
                "tool_names": [tool.name for tool in tools or []],
                "tool_choice": tool_choice.model_dump(mode="json") if tool_choice else None,
            }
        )
        if not self._responses:
            raise AssertionError("Unexpected LLM call; add another scripted response.")
        return self._responses.pop(0)


class FailingGraph:
    async def ainvoke(self, state: dict[str, Any], config: dict[str, Any] | None = None):
        del state, config
        raise RuntimeError("graph failed")


class SequenceClock:
    def __init__(self, values):
        self._values = list(values)

    def __call__(self):
        if not self._values:
            raise AssertionError("Unexpected clock call.")
        return self._values.pop(0)


@asynccontextmanager
async def fake_session_context() -> AsyncGenerator[AsyncSession]:
    yield cast(AsyncSession, object())


def build_app(graph, path, clock_values=None):
    app = FastAPI()
    app.state.agent_graph = graph
    app.state.emit_run_event = make_jsonl_emitter(path)
    if clock_values is not None:
        app.state.run_event_clock = SequenceClock(clock_values)
    app.include_router(agent_router, prefix="/agent")
    return app


original_request_session = agent_api._request_session
original_invoke_tool_binding = graph_module.invoke_tool_binding
original_submit_work_order = graph_module.submit_work_order

agent_api._request_session = fake_session_context

In [5]:
def tool_call(name, input=None, id_suffix="1"):
    return ToolCallRequest(id=f"{name}-{id_suffix}", name=name, input=input or {})


def interpret_response(intent, asset_identifier):
    return LLMResponse(
        tool_calls=[
            tool_call(
                "interpret_request",
                {"intent": intent, "asset_identifier": asset_identifier},
            )
        ]
    )


def evidence_response(tool_name, *, policy_type="recurring_fault"):
    if tool_name == "search_maintenance_docs":
        input = {"query": "maintenance docs"}
    elif tool_name == "get_plant_policy":
        input = {"policy_type": policy_type}
    else:
        input = {}
    return LLMResponse(tool_calls=[tool_call(tool_name, input)])


def draft_response():
    return LLMResponse(
        tool_calls=[
            tool_call(
                "create_work_order_draft",
                {
                    "issue": "Recurring bearing overheating",
                    "recommended_action": "Investigate lubrication, alignment, and bearing condition.",
                    "priority": "high",
                },
            )
        ]
    )


def synthesis_response(evidence_id="DOC-03"):
    return LLMResponse(
        tool_calls=[
            tool_call(
                "synthesize_response",
                {
                    "answer": "Use the gathered evidence to inspect the asset.",
                    "confidence": "hypothesis",
                    "evidence_used": [evidence_id],
                },
            )
        ]
    )

In [6]:
def asset(asset_id):
    return AssetRecord(
        asset_id=asset_id,
        asset_type="centrifugal_pump",
        model="CP-200",
        location="Line 3",
        installation_date=date(2021, 6, 1),
        status="operational",
    )


def classified_reading():
    return ClassifiedReading(
        source_id="TS-001",
        metric="bearing_temperature_c",
        value=Decimal("78.0"),
        unit="C",
        tier="normal",
        operating_limit_id="OL-002",
        rule_text="Normal < 82; high >= 82",
    )


def doc_hit():
    return DocSearchHit(
        chunk_id="DOC-03-C1",
        document_id="DOC-03",
        section="Mechanical seal inspection",
        page="1",
        topic="seal inspection",
        manufacturer="Synthetic",
        source_product_family="CP",
        applicability="PUMP-103",
        source_url="synthetic://DOC-03",
        content_provenance="synthetic",
        linked_fault_codes=["F102"],
        evidence_text="Inspect the seal and bearing assembly.",
        similarity_score=0.9,
    )


async def fake_invoke_tool_binding(
    tool_name: str,
    args: dict[str, object],
    state: GraphState,
    session: AsyncSession,
) -> object:
    del session
    if tool_name == "resolve_asset":
        identifier = cast(str, args["identifier"])
        if identifier == "PUMP-999":
            return ResolveAssetResult(status="not_found")
        return ResolveAssetResult(status="resolved", asset=asset(identifier))
    if tool_name == "get_asset_status":
        return GetAssetStatusResult(
            asset=asset("PUMP-103"),
            telemetry=None,
            classified_readings=[classified_reading()],
        )
    if tool_name == "get_maintenance_history":
        return GetMaintenanceHistoryResult(asset=asset("PUMP-103"))
    if tool_name == "search_maintenance_docs":
        return SearchMaintenanceDocsResult(query="maintenance docs", results=[doc_hit()])
    if tool_name == "get_plant_policy":
        return GetPlantPolicyResult(policy_type=cast(str, args["policy_type"]))
    if tool_name == "create_work_order_draft":
        return WorkOrderDraft(
            draft_id=state.get("request_id", "notebook-request"),
            asset_id="PUMP-103",
            issue=cast(str, args["issue"]),
            recommended_action=cast(str, args["recommended_action"]),
            priority=cast(Literal["low", "high"], args["priority"]),
            supporting_evidence=[],
        )
    raise AssertionError(f"Unexpected tool call: {tool_name}")


async def fake_submit_work_order(
    draft: WorkOrderDraft,
    *,
    approval_status: str,
    session: AsyncSession,
) -> WorkOrderRecord:
    del session
    assert approval_status == "approved"
    return WorkOrderRecord(
        work_order_id="WO-NOTEBOOK-001",
        asset_id=draft.asset_id,
        issue=draft.issue,
        priority=draft.priority,
        status="submitted",
        created_at=date(2026, 8, 21),
        approved=True,
    )


graph_module.invoke_tool_binding = fake_invoke_tool_binding
graph_module.submit_work_order = fake_submit_work_order

## 4. Scenario A: normal query emits one JSONL event

This run completes normally. The event's `run_id` matches the API response `request_id`, the `tool_calls` list is sorted by sequence, and `final_output` stores the full returned `AgentQueryResponse`.

In [7]:
normal_path = reset_event_file("normal-query")
normal_llm = ScriptedLLMClient(
    [
        interpret_response("troubleshooting", "PUMP-102"),
        evidence_response("get_asset_status"),
        evidence_response("search_maintenance_docs"),
        evidence_response("get_maintenance_history"),
        LLMResponse(tool_calls=[]),
        synthesis_response("DOC-03"),
    ]
)
normal_app = build_app(
    build_agent_graph(AgentGraphDependencies(llm_client=normal_llm)),
    normal_path,
    [
        datetime(2026, 8, 21, 10, 0, 0, tzinfo=UTC),
        datetime(2026, 8, 21, 10, 0, 0, 375000, tzinfo=UTC),
    ],
)

async with AsyncClient(transport=ASGITransport(app=normal_app), base_url="http://testserver") as client:
    normal_response = await client.post(
        "/agent/query",
        json={"query": "PUMP-102 is vibrating much more than usual. What could be wrong?"},
    )

print_json({"status_code": normal_response.status_code, "body": normal_response.json()})
normal_events = print_event_table(normal_path)

{
  "status_code": 200,
  "body": {
    "request_id": "6a414c6a-85b3-4bab-95ac-b01f359b4b96",
    "status": "ok",
    "asset_id": "PUMP-102",
    "answer": "Use the gathered evidence to inspect the asset.",
    "confidence": "hypothesis",
    "evidence_used": [
      "DOC-03"
    ],
    "structured_evidence": [
      {
        "source": "ClassifiedReading",
        "source_type": "telemetry_snapshot",
        "source_id": "TS-001",
        "summary": "source_type='telemetry_snapshot' source_id='TS-001' metric='bearing_temperature_c' value=Decimal('78.0') unit='C' tier='normal' operating_limit_id='OL-002' rule_text='Normal < 82; high >= 82'",
        "reference_id": "TS-001"
      }
    ],
    "document_evidence": [
      {
        "document_id": "DOC-03",
        "section": "Mechanical seal inspection",
        "excerpt": "Inspect the seal and bearing assembly."
      }
    ],
    "pending_action": null,
    "error": null
  }
}
[
  {
    "event_id": "41542afd-bdaf-4d5b-877d-9af7650024b

## 5. Scenario B: unknown asset still emits one completed event

A business outcome like `unknown_asset` is not a telemetry failure. It produces one completed `RunEvent` with the same public response embedded in `final_output`.

In [8]:
unknown_path = reset_event_file("unknown-asset")
unknown_llm = ScriptedLLMClient([interpret_response("troubleshooting", "PUMP-999")])
unknown_app = build_app(
    build_agent_graph(AgentGraphDependencies(llm_client=unknown_llm)),
    unknown_path,
)

async with AsyncClient(transport=ASGITransport(app=unknown_app), base_url="http://testserver") as client:
    unknown_response = await client.post(
        "/agent/query",
        json={"query": "PUMP-999 has high vibration. Diagnose it."},
    )

print_json({"status_code": unknown_response.status_code, "body": unknown_response.json()})
unknown_events = print_event_table(unknown_path)

{
  "status_code": 200,
  "body": {
    "request_id": "f085a62f-e921-48a4-92dd-3884b01c4d89",
    "status": "unknown_asset",
    "asset_id": null,
    "answer": "I couldn't find an asset matching 'PUMP-999'. Please provide a valid asset ID.",
    "confidence": null,
    "evidence_used": [],
    "structured_evidence": [],
    "document_evidence": [],
    "pending_action": null,
    "error": null
  }
}
[
  {
    "event_id": "4895be14-26b5-490a-9bb1-259572ac9123",
    "run_id": "f085a62f-e921-48a4-92dd-3884b01c4d89",
    "emitted_at": "2026-08-21T12:42:48.229816+00:00",
    "latency_ms": 8,
    "status": "unknown_asset",
    "request": "PUMP-999 has high vibration. Diagnose it.",
    "tool_calls": [
      {
        "tool_name": "resolve_asset",
        "sequence": 1
      }
    ],
    "final_output_status": "unknown_asset",
    "final_output_request_id": "f085a62f-e921-48a4-92dd-3884b01c4d89",
    "pending_action": null,
    "error": null
  }
]


## 6. Scenario C: HITL pause and approval emit two correlated events

Phase 7 records one event for the pause response and one event for the approval resume response. They share `run_id` because the Phase 6 draft id, thread id, and request id are intentionally unified, but each event has its own `event_id`.

In [9]:
hitl_path = reset_event_file("hitl-approve")
hitl_llm = ScriptedLLMClient(
    [
        interpret_response("work_order_request", "PUMP-103"),
        evidence_response("get_asset_status"),
        evidence_response("get_maintenance_history"),
        evidence_response("search_maintenance_docs"),
        evidence_response("get_plant_policy", policy_type="consequential_action"),
        draft_response(),
    ]
)
hitl_app = build_app(build_agent_graph(AgentGraphDependencies(llm_client=hitl_llm)), hitl_path)

async with AsyncClient(transport=ASGITransport(app=hitl_app), base_url="http://testserver") as client:
    pause_response = await client.post(
        "/agent/query",
        json={"query": "Create and submit a work order for PUMP-103."},
    )
    draft_id = pause_response.json()["pending_action"]["draft_id"]
    approve_response = await client.post(
        f"/agent/approvals/{draft_id}",
        json={"decision": "approve"},
    )

print_json(
    {
        "pause_status_code": pause_response.status_code,
        "pause_body": pause_response.json(),
        "approval_status_code": approve_response.status_code,
        "approval_body": approve_response.json(),
    }
)
hitl_events = print_event_table(hitl_path)
print_json(
    {
        "same_run_id": hitl_events[0].run_id == hitl_events[1].run_id == draft_id,
        "distinct_event_ids": hitl_events[0].event_id != hitl_events[1].event_id,
        "requests_recorded": [event.request for event in hitl_events],
    }
)

{
  "pause_status_code": 200,
  "pause_body": {
    "request_id": "2d2de42b-531c-4fda-a786-afe9d8281459",
    "status": "needs_approval",
    "asset_id": "PUMP-103",
    "answer": "Work order draft 2d2de42b-531c-4fda-a786-afe9d8281459 for PUMP-103 needs approval. Issue: Recurring bearing overheating. Priority: high. Recommended action: Investigate lubrication, alignment, and bearing condition.",
    "confidence": null,
    "evidence_used": [],
    "structured_evidence": [
      {
        "source": "ClassifiedReading",
        "source_type": "telemetry_snapshot",
        "source_id": "TS-001",
        "summary": "source_type='telemetry_snapshot' source_id='TS-001' metric='bearing_temperature_c' value=Decimal('78.0') unit='C' tier='normal' operating_limit_id='OL-002' rule_text='Normal < 82; high >= 82'",
        "reference_id": "TS-001"
      }
    ],
    "document_evidence": [
      {
        "document_id": "DOC-03",
        "section": "Mechanical seal inspection",
        "excerpt": "I

## 7. Scenario D: HITL pause and rejection emit two correlated events

Rejection is still a successful completion of the pending workflow from the API's perspective. The resume event records `request='reject'`, shares the original `run_id`, and does not add submitted work-order evidence to `final_output.structured_evidence`.

In [10]:
reject_path = reset_event_file("hitl-reject")
reject_llm = ScriptedLLMClient(
    [
        interpret_response("work_order_request", "PUMP-103"),
        evidence_response("get_asset_status"),
        evidence_response("get_maintenance_history"),
        evidence_response("search_maintenance_docs"),
        evidence_response("get_plant_policy", policy_type="consequential_action"),
        draft_response(),
    ]
)
reject_app = build_app(build_agent_graph(AgentGraphDependencies(llm_client=reject_llm)), reject_path)

async with AsyncClient(transport=ASGITransport(app=reject_app), base_url="http://testserver") as client:
    reject_pause_response = await client.post(
        "/agent/query",
        json={"query": "Create a work order draft for recurring bearing overheating on PUMP-103."},
    )
    reject_draft_id = reject_pause_response.json()["pending_action"]["draft_id"]
    reject_response = await client.post(
        f"/agent/approvals/{reject_draft_id}",
        json={"decision": "reject"},
    )

print_json(
    {
        "pause_status_code": reject_pause_response.status_code,
        "pause_body": reject_pause_response.json(),
        "rejection_status_code": reject_response.status_code,
        "rejection_body": reject_response.json(),
    }
)
reject_events = print_event_table(reject_path)
print_json(
    {
        "same_run_id": reject_events[0].run_id == reject_events[1].run_id == reject_draft_id,
        "distinct_event_ids": reject_events[0].event_id != reject_events[1].event_id,
        "requests_recorded": [event.request for event in reject_events],
        "resume_evidence_source_types": [
            item.source_type for item in reject_events[1].final_output.structured_evidence
        ],
        "work_order_evidence_added": "work_order"
        in {item.source_type for item in reject_events[1].final_output.structured_evidence},
    }
)

{
  "pause_status_code": 200,
  "pause_body": {
    "request_id": "bc2801e2-5bc0-4513-a980-172d3e718433",
    "status": "needs_approval",
    "asset_id": "PUMP-103",
    "answer": "Work order draft bc2801e2-5bc0-4513-a980-172d3e718433 for PUMP-103 needs approval. Issue: Recurring bearing overheating. Priority: high. Recommended action: Investigate lubrication, alignment, and bearing condition.",
    "confidence": null,
    "evidence_used": [],
    "structured_evidence": [
      {
        "source": "ClassifiedReading",
        "source_type": "telemetry_snapshot",
        "source_id": "TS-001",
        "summary": "source_type='telemetry_snapshot' source_id='TS-001' metric='bearing_temperature_c' value=Decimal('78.0') unit='C' tier='normal' operating_limit_id='OL-002' rule_text='Normal < 82; high >= 82'",
        "reference_id": "TS-001"
      }
    ],
    "document_evidence": [
      {
        "document_id": "DOC-03",
        "section": "Mechanical seal inspection",
        "excerpt": "I

## 8. Scenario E: internal error emits one error event

The route converts an unhandled graph exception into the public error response, then emits exactly one `RunEvent` whose `error` envelope matches `final_output.error`.

In [11]:
error_path = reset_event_file("internal-error")
error_app = build_app(
    FailingGraph(),
    error_path,
    [
        datetime(2026, 8, 21, 11, 0, 0, tzinfo=UTC),
        datetime(2026, 8, 21, 11, 0, 0, 125000, tzinfo=UTC),
    ],
)

async with AsyncClient(transport=ASGITransport(app=error_app), base_url="http://testserver") as client:
    error_response = await client.post(
        "/agent/query",
        json={"query": "Check pump vibration.", "asset_id": "PUMP-103"},
    )

print_json({"status_code": error_response.status_code, "body": error_response.json()})
error_events = print_event_table(error_path)
print_json(
    {
        "error_matches_final_output": error_events[0].error == error_events[0].final_output.error,
        "error_code": error_events[0].error.code if error_events[0].error else None,
    }
)

{
  "status_code": 200,
  "body": {
    "request_id": "9e7f1471-12bc-4b56-9821-4149768fb945",
    "status": "error",
    "asset_id": "PUMP-103",
    "answer": null,
    "confidence": null,
    "evidence_used": [],
    "structured_evidence": [],
    "document_evidence": [],
    "pending_action": null,
    "error": {
      "code": "unhandled_exception",
      "message": "graph failed"
    }
  }
}
[
  {
    "event_id": "a73783b0-cf88-451a-ab33-bf5a25a2b6ca",
    "run_id": "9e7f1471-12bc-4b56-9821-4149768fb945",
    "emitted_at": "2026-08-21T11:00:00.125000+00:00",
    "latency_ms": 125,
    "status": "error",
    "request": "Check pump vibration.",
    "tool_calls": [],
    "final_output_status": "error",
    "final_output_request_id": "9e7f1471-12bc-4b56-9821-4149768fb945",
    "pending_action": null,
    "error": {
      "code": "unhandled_exception",
      "message": "graph failed"
    }
  }
]
{
  "error_matches_final_output": true,
  "error_code": "unhandled_exception"
}


## 9. Inspect raw JSONL lines

The companion reader validates each line back into `RunEvent`, but sometimes it is useful to inspect the raw serialized JSON. Change `path_to_inspect` to any scenario path above.

In [12]:
path_to_inspect = hitl_path
print(path_to_inspect)
print(path_to_inspect.read_text(encoding="utf-8"))

/home/sermengi/industrial-maintenance-agent/notebooks/.phase7-run-events/hitl-approve.jsonl
{"event_id":"2dbb05c7-0be9-4cdb-8680-3a3d0031e8a7","run_id":"2d2de42b-531c-4fda-a786-afe9d8281459","emitted_at":"2026-08-21T12:42:48.271876Z","latency_ms":19,"status":"needs_approval","request":"Create and submit a work order for PUMP-103.","tool_calls":[{"tool_name":"resolve_asset","sequence":1},{"tool_name":"get_asset_status","sequence":2},{"tool_name":"get_maintenance_history","sequence":3},{"tool_name":"search_maintenance_docs","sequence":4},{"tool_name":"get_plant_policy","sequence":5},{"tool_name":"create_work_order_draft","sequence":6}],"final_output":{"request_id":"2d2de42b-531c-4fda-a786-afe9d8281459","status":"needs_approval","asset_id":"PUMP-103","answer":"Work order draft 2d2de42b-531c-4fda-a786-afe9d8281459 for PUMP-103 needs approval. Issue: Recurring bearing overheating. Priority: high. Recommended action: Investigate lubrication, alignment, and bearing condition.","confidence":nu

## 10. Optional cleanup

Restore the patched functions if you keep using the same notebook kernel for other experiments.

In [13]:
agent_api._request_session = original_request_session
graph_module.invoke_tool_binding = original_invoke_tool_binding
graph_module.submit_work_order = original_submit_work_order

print(f"Run-event files remain available under: {RUN_EVENTS_DIR}")

Run-event files remain available under: /home/sermengi/industrial-maintenance-agent/notebooks/.phase7-run-events
